# Radyolog-AI — Raporlama v3 (İngilizce JSON deneyi)

Bu defter 80 vaka × 5 rapor üreticisini (**Template, Qwen, Gemma, Llama, Claude**) ve ek kol olarak **Claude-thinking**'i yeni deney düzeniyle üretir ve radyologla kıyaslar.

**Deney düzeni:** tüm modellere aynı İngilizce klinik JSON, aynı talimat (tek kullanıcı mesajı) ve aynı çıktı token sınırı (3500). Bildirilen farklar: Claude'da örnekleme ayarlanamaz (API varsayılanı) ve seed yok; açık modellerde T=1.0, top_p=1.0, top_k kapalı, seed 42, 4-bit NF4. Template bir **kural tabanlı kontroldür**, BDM değildir.

**Çıktı klasörü:** `Radyolog-AI-2/reports_en_json_v3` — eski `reports` klasörüne **dokunulmaz**.

### Eski koşularda yaşanan hatalar ve bu defterdeki önlemler
| Eski hata | Bu defterde |
|---|---|
| `git pull --ff-only` sessizce başarısız oldu, **eski kod** koştu | `fetch` + `reset --hard`, commit yazdırılır, sürüm ve dosya **assert** edilir |
| Aynı kernel'de eski modül bellekte kaldı (`AttributeError`) | Her model **ayrı süreçte** (`!python compare_llm.py`) koşar |
| "TUM MODELLER BITTI" yazdı ama tüm modeller hata vermişti | Her modelden sonra **sayım kontrolü**; 80 değilse açıkça uyarır |
| API anahtarı adları yanlış / hücre çalıştırılmamış | Anahtarlar `VAR/YOK` diye kontrol edilir; eksikse durur |
| Geçersiz HF token hücreyi çökertti; kapılı modeller sonra hata verdi | Token ve **Gemma/Llama erişimi koşudan önce** denetlenir |
| Raporları silen "cache temizle" hücresi | **Silme hücresi yok.** Yeni deney ayrı klasörde |
| Gemini kota duvarı (6/80) | Gemini bu deneyde **yok** |
| Türkçe değerler, Claude sıcaklığı, seed, faz bilgisi | Kodda düzeltildi; **koşul testleri** koşudan önce çalıştırılır |
| Claude parası doğrulanmadan harcandı | Claude **en sonda**; önce **1 vaka deneme** + maliyet tahmini + açık onay |
| Drive bağlantısı koptu, yollar boş döndü | Her ağır hücre başında Drive kontrolü |
| Paket sürümleri kayıtsız | Sürümler ve commit deney klasörüne yazılır |

Hücreleri **sırayla** çalıştır. Bir kontrol `AssertionError` verirse **devam etme**.

## 0) GPU ve Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
BASE = '/content/drive/MyDrive/Radyolog-AI-2'
EXP  = f'{BASE}/reports_en_json_v3'
REPO = '/content/Radyolog-AI'
BRANCH = 'claude/radyolog-ai-proje-dokumani-764226'
!nvidia-smi -L
!df -h /content | tail -1
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert gpu.returncode == 0 and 'GPU' in gpu.stdout, (
    'GPU YOK — CPU çalışma zamanındasın. Çalışma zamanı > Çalışma zamanı türünü değiştir > '
    'T4 GPU > KAYDET, sonra defteri baştan çalıştır.')

import glob
assert os.path.isdir(BASE), 'Drive klasörü yok: ' + BASE
fnd = [p for p in glob.glob(f'{BASE}/findings/*.json') if not p.endswith('.meta.json')]
print('bulgu dosyası:', len(fnd))
assert len(fnd) == 80, 'findings/ altında 80 bulgu dosyası bekleniyordu'
assert os.path.exists(f'{BASE}/reports_final.xlsx'), 'reports_final.xlsx yok'
assert os.path.exists(f'{BASE}/protocol/kohort_80_liste.csv'), 'kohort_80_liste.csv yok'
print('Drive OK')

## 1) Kod: doğru dal, doğru commit

In [ ]:
if not os.path.exists(REPO):
    !git clone -q https://github.com/temelinho/Radyolog-AI.git {REPO}
!cd {REPO} && git fetch -q origin && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!cd {REPO} && git log --oneline -1
COMMIT = subprocess.check_output(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD']).decode().strip()
v = subprocess.check_output(['python', '-c', 'from pipeline import config; print(config.REPORT_PROMPT_VERSION, config.PROMPT_VERSION)'],
                            cwd=REPO).decode().split()
print('commit:', COMMIT, '| rapor istem sürümü:', v[0], '| ölçüm sürümü:', v[1])
assert v == ['3.0.0', '2.0.0'], 'YANLIŞ KOD SÜRÜMÜ — hücreyi tekrar çalıştır'
assert os.path.exists(f'{REPO}/tests/test_english_json.py'), 'Yeni testler yok — eski kod'
print('Kod OK')

## 2) Kurulum ve sürüm kaydı
`protobuf ... incompatible` uyarıları zararsızdır.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes anthropic pytest openpyxl
import json, sys
vers = {}
for m in ['torch', 'transformers', 'accelerate', 'bitsandbytes', 'anthropic', 'huggingface_hub']:
    out = subprocess.run([sys.executable, '-c', f'import {m}; print({m}.__version__)'],
                         capture_output=True, text=True)
    vers[m] = out.stdout.strip() or 'YOK'
print(json.dumps(vers, indent=2))
assert 'YOK' not in vers.values(), 'Eksik paket var'

## 3) Koşul testleri (koşudan ÖNCE)
Aynı girdi, İngilizce JSON, önbellek izolasyonu, şablonun kontrol olması ve ayıklayıcı testleri.

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_same_conditions.py',
                    'tests/test_english_json.py', 'tests/test_compare_reference.py'],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])
assert r.returncode == 0, 'TESTLER BAŞARISIZ — koşma: ' + r.stdout[-3000:]
print('Testler OK')

## 4) Anahtarlar ve model erişimi
Colab Secrets: `ANTHROPIC_API_KEY`, `HF_TOKEN`. Değerler yazdırılmaz.

In [ ]:
from google.colab import userdata
for k in ['ANTHROPIC_API_KEY', 'HF_TOKEN']:
    try:
        os.environ[k] = userdata.get(k) or ''
    except Exception:
        os.environ[k] = ''
    print(k, 'VAR' if os.environ[k] else 'YOK')
assert os.environ['ANTHROPIC_API_KEY'] and os.environ['HF_TOKEN'], 'Eksik anahtar'

from huggingface_hub import login, hf_hub_download
login(os.environ['HF_TOKEN'], add_to_git_credential=False)
for repo in ['Qwen/Qwen2.5-14B-Instruct', 'google/gemma-2-9b-it', 'meta-llama/Llama-3.1-8B-Instruct']:
    try:
        hf_hub_download(repo, 'config.json')
        print('erişim OK  :', repo)
    except Exception as e:
        print('ERİŞİM YOK:', repo, '->', type(e).__name__)
        raise

## 5) Deney klasörü ve modele giden girdinin önizlemesi

In [ ]:
os.environ['RADYOLOG_BASE'] = BASE
os.environ['RADYOLOG_REPORTS_DIR'] = EXP
os.environ.pop('RADYOLOG_REPORTS', None)
os.makedirs(EXP, exist_ok=True)
print('deney klasörü:', EXP, '| mevcut rapor:', len(glob.glob(f'{EXP}/*.txt')))

PREVIEW = r'''
import glob, json
from pipeline import config, reporting as R
assert str(config.LAYOUT.reports).endswith("reports_en_json_v3"), config.LAYOUT.reports
p = sorted(x for x in glob.glob(str(config.LAYOUT.findings) + "/*.json") if not x.endswith(".meta.json"))[0]
d = json.load(open(p, encoding="utf-8")); f = d.get("bulgular", d)
msg = R.build_input(R.SYSTEM_PROMPT, R.build_prompt(f))
print(msg)
import re
assert not re.search(r"karaci|pankreas|portal ven(?!o)|\bvci\b", msg, re.I), "TÜRKÇE DEĞER VAR"
print("\n--- girdi İngilizce, OK")
'''
r = subprocess.run([sys.executable, '-c', PREVIEW], cwd=REPO, capture_output=True, text=True, env=os.environ)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0

## 6) Yardımcılar: model koşturma ve sayım kontrolü

In [ ]:
def kos(model, dosya=None):
    # Her model AYRI süreçte: eski modül/VRAM kalıntısı olmaz. Çıktı Drive'a log olarak da yazılır.
    arg = f'"{dosya}"' if dosya else ''
    !cd {REPO} && RADYOLOG_MODELS={model} python compare_llm.py {arg} 2>&1 | tee -a "{EXP}/log_{model}.txt" | tail -n 25

def kontrol(model, beklenen=80):
    # Geçerli rapor + kayıtlı geçersiz çıktı = denenmiş vaka. Geçersizler TEKRAR denenmez.
    txt = sorted(glob.glob(f'{EXP}/*__{model}.txt'))
    bad = sorted(glob.glob(f'{EXP}/*__{model}.gecersiz.json'))
    eksik_kayit = [t for t in txt if not os.path.exists(t[:-4] + '.uretim.json')]
    print(f'{model}: geçerli {len(txt)} | geçersiz {len(bad)} | denenen {len(txt) + len(bad)}/{beklenen}'
          f' | üretim kaydı eksik: {len(eksik_kayit)}')
    for b in bad:
        print('  geçersiz:', os.path.basename(b).rsplit('__', 1)[0], '->', json.load(open(b, encoding='utf-8'))['neden'])
    if len(txt) + len(bad) != beklenen:
        tum = {os.path.basename(p)[:-5] for p in fnd}
        var = {os.path.basename(t).rsplit('__', 1)[0] for t in txt + bad}
        print('  DENENMEMİŞ vakalar:', sorted(tum - var))
    return len(txt) + len(bad) == beklenen and not eksik_kayit

## 7) Template (kural tabanlı kontrol)

In [ ]:
kos('Template'); assert kontrol('Template')
print(open(sorted(glob.glob(f'{EXP}/*__Template.txt'))[0], encoding='utf-8').read())

## 8) Açık modeller
Her biri ilk çalıştırmada indirilir (Qwen ~30 GB, Gemma ~18 GB, Llama ~16 GB). Koşu kesilirse hücreyi tekrar çalıştır: biten raporlar atlanır.

In [ ]:
assert subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0, 'GPU YOK — durdur'
kos('Qwen');  print('Qwen OK' if kontrol('Qwen') else 'Qwen EKSİK — hücreyi tekrar çalıştır')

In [ ]:
!rm -rf /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct
kos('Gemma'); print('Gemma OK' if kontrol('Gemma') else 'Gemma EKSİK — hücreyi tekrar çalıştır')

In [ ]:
!rm -rf /root/.cache/huggingface/hub/models--google--gemma-2-9b-it
kos('Llama'); print('Llama OK' if kontrol('Llama') else 'Llama EKSİK — hücreyi tekrar çalıştır')

## 9) Claude — önce 1 vaka deneme ve maliyet tahmini
Düşünme kapalı olmalı (`thinking_blocks: false`) ve bitiş nedeni `end_turn` olmalı.

In [ ]:
ilk = sorted(fnd)[0]
kos('Claude', ilk)
ad = os.path.basename(ilk)[:-5]
k = json.load(open(f'{EXP}/{ad}__Claude.uretim.json', encoding='utf-8'))
print(json.dumps({x: k[x] for x in ('finish_reason', 'usage')}, indent=2))
assert k['finish_reason'] == 'end_turn', 'Beklenmeyen bitiş nedeni'
assert k['usage'].get('thinking_blocks') is False, 'Claude DÜŞÜNDÜ — koşul tutmuyor, durdur'
tin, tout = k['usage']['in'], k['usage']['out']
tahmin = 80 * (tin * 2 + tout * 10) / 1e6   # Sonnet 5.5: 2 $/M girdi, 10 $/M çıktı
print(f'1 vaka: {tin} girdi + {tout} çıktı token  ->  80 vaka için tahmini ~{tahmin:.2f} $')

## 10) Claude — 80 vaka
Bakiyeyi kontrol et, sonra `CLAUDE_ONAY = True` yap.

In [ ]:
CLAUDE_ONAY = False   # <- bakiyeyi kontrol ettikten sonra True yap
assert CLAUDE_ONAY, 'Onay verilmedi'
kos('Claude'); print('Claude OK' if kontrol('Claude') else 'Claude EKSİK — hücreyi tekrar çalıştır')

## 10b) Claude-thinking — EK KOL (düşünme açık, sağlayıcı varsayılanı)
Ana karşılaştırmaya girmez; "düşünme ne değiştirdi?" sorusu için. Aynı girdi, düşünme açık, daha geniş token sınırı (düşünme tokenleri de sayılır). Önce 1 vaka deneme ve maliyet tahmini.

In [ ]:
kos('Claude-thinking', ilk)
k = json.load(open(f'{EXP}/{ad}__Claude-thinking.uretim.json', encoding='utf-8'))
print(json.dumps({x: k[x] for x in ('finish_reason', 'usage')}, indent=2))
assert k['finish_reason'] == 'end_turn', 'Beklenmeyen bitiş nedeni'
tin, tout = k['usage']['in'], k['usage']['out']
tahmin = 80 * (tin * 2 + tout * 10) / 1e6
print(f'1 vaka: {tin} girdi + {tout} çıktı (düşünme dahil) ->  80 vaka için tahmini ~{tahmin:.2f} $')

In [ ]:
CLAUDE_THINKING_ONAY = False   # <- bakiyeyi kontrol ettikten sonra True yap
assert CLAUDE_THINKING_ONAY, 'Onay verilmedi'
kos('Claude-thinking'); print('Claude-thinking OK' if kontrol('Claude-thinking') else 'EKSİK — hücreyi tekrar çalıştır')

## 11) Doğrulama — koşullar gerçekten tuttu mu?

In [ ]:
import re, collections
kos_j = json.load(open(f'{EXP}/kosullar.json', encoding='utf-8'))
print('istem sürümü:', kos_j['prompt_version'], '| sistem istemi SHA:', kos_j['system_prompt_sha256'][:16])
for m, c in kos_j['modeller'].items():
    print(f"  {m:9} input={c['input_format']} | thinking={c['thinking']} | max_tokens={c['max_tokens']} | sampling={c['sampling']}")

sorun = collections.Counter()
for m in ['Template', 'Qwen', 'Gemma', 'Llama', 'Claude', 'Claude-thinking']:
    ok = kontrol(m)
    girdiler = set()
    for t in glob.glob(f'{EXP}/*__{m}.txt'):
        k = json.load(open(t[:-4] + '.uretim.json', encoding='utf-8'))
        girdiler.add(k['conditions']['prompt_version'])
        if m == 'Claude' and k['usage'].get('thinking_blocks'):
            sorun['claude_dusundu'] += 1
        if re.search(r'karaci|pankreas', open(t, encoding='utf-8').read(), re.I):
            sorun[f'{m}_turkce_organ'] += 1
    print(f'  {m}: istem sürümleri {girdiler}')
    if not ok: sorun[f'{m}_eksik'] += 1
print('SORUNLAR:', dict(sorun) or 'yok')
assert not sorun, 'Doğrulama başarısız'

## 12) Sayısal sadakat özeti

In [ ]:
import pandas as pd
df = pd.read_csv(f'{EXP}/ozet.csv')
print('satır:', len(df))
df['oran'] = pd.to_numeric(df['oran'], errors='coerce')
df['gecersiz'] = df['yontem'].astype(str).str.startswith('gecersiz')
print(df.groupby('model').agg(olculebilir=('oran', 'count'), ortalama=('oran', 'mean'),
                              medyan=('oran', 'median'), gecersiz_rapor=('gecersiz', 'sum')).round(3))

## 13) Radyolog kıyası
Çıktı: `outputs/reports_en_json_v3` (eski `outputs` ezilmez).

In [ ]:
!cd {REPO} && python compare_reference.py --reports-xlsx "{BASE}/reports_final.xlsx" 2>&1 | tee "{EXP}/log_compare_reference.txt" | sed -n '/1) TESP/,$p'

## 14) Çalıştırma kaydı

In [ ]:
import datetime
bilgi = {'tarih': datetime.datetime.now().isoformat(timespec='seconds'), 'commit': COMMIT,
         'paketler': vers, 'deney_klasoru': EXP}
json.dump(bilgi, open(f'{EXP}/calistirma_bilgisi.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print(json.dumps(bilgi, indent=2, ensure_ascii=False))